# Content-Based Recommender Using User Profiles and Course Genres

This notebook implements the content-based method described in the presentation: compare each user's genre preference vector with each unseen course's genre vector.

In [1]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

DATA_DIR = Path("datasets")
DATA_URLS = {
    "ratings.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv",
    "course_genre.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv",
    "rs_content_test.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv",
    "user_profile.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_profile.csv",
    "course_processed.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv",
    "courses_bows.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv",
    "sim.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/sim.csv",
}

def ensure_dataset(filename):
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(DATA_URLS[filename], path)
    return path

def load_csv(filename, **kwargs):
    return pd.read_csv(ensure_dataset(filename), **kwargs)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

In [2]:
course_genres_df = load_csv("course_genre.csv")
profile_df = load_csv("user_profile.csv")
test_users_df = load_csv("rs_content_test.csv")

genre_cols = [col for col in course_genres_df.columns if col not in ["COURSE_ID", "TITLE"]]
all_courses = set(course_genres_df["COURSE_ID"])
test_user_ids = sorted(test_users_df["user"].unique())

print("Course genre matrix:", course_genres_df.shape)
print("User profile matrix:", profile_df.shape)
print("Test interactions:", test_users_df.shape)

Course genre matrix: (307, 16)
User profile matrix: (33901, 15)
Test interactions: (9402, 3)


## Recommendation Logic

For each test user:

1. Build the user's genre preference vector.
2. Remove courses the user has already taken.
3. Score every unseen course with a dot product between user profile and course genre vector.
4. Keep courses with score greater than or equal to `10.0`.
5. Rank the remaining courses by score.

In [3]:
SCORE_THRESHOLD = 10.0

def recommend_for_user(user_id, top_n=None):
    user_profile = profile_df.loc[profile_df["user"] == user_id, genre_cols]
    if user_profile.empty:
        return pd.DataFrame(columns=["USER", "COURSE_ID", "TITLE", "SCORE"])

    user_vector = user_profile.iloc[0].to_numpy(dtype=float)
    enrolled_courses = set(test_users_df.loc[test_users_df["user"] == user_id, "item"])
    candidate_df = course_genres_df.loc[~course_genres_df["COURSE_ID"].isin(enrolled_courses)].copy()
    candidate_matrix = candidate_df[genre_cols].to_numpy(dtype=float)

    candidate_df["SCORE"] = candidate_matrix.dot(user_vector)
    recommendations = (
        candidate_df.loc[candidate_df["SCORE"] >= SCORE_THRESHOLD, ["COURSE_ID", "TITLE", "SCORE"]]
        .sort_values("SCORE", ascending=False)
    )
    if top_n is not None:
        recommendations = recommendations.head(top_n)
    recommendations.insert(0, "USER", user_id)
    return recommendations.reset_index(drop=True)

sample_user = test_user_ids[0]
recommend_for_user(sample_user, top_n=10)

,USER,COURSE_ID,TITLE,SCORE
0,37465,RP0105EN,analyzing big data in r using apache spark,27.0
1,37465,SC0103EN,spark overview for scala analytics,27.0
2,37465,TMP0105EN,getting started with the data apache spark ma...,27.0
3,37465,excourse73,analyzing big data with sql,27.0
4,37465,excourse31,cloud computing applications part 2 big data...,27.0
5,37465,BD0212EN,spark fundamentals ii,27.0
6,37465,excourse72,foundations for big data analysis with sql,27.0
7,37465,excourse70,big data capstone project,24.0
8,37465,excourse03,nosql systems,24.0
9,37465,BD0141EN,accessing hadoop data using hive,24.0


In [4]:
recommendation_frames = [recommend_for_user(user_id) for user_id in test_user_ids]
recommendations_df = pd.concat(recommendation_frames, ignore_index=True)

users_with_recommendations = recommendations_df["USER"].nunique()
avg_recommendations_for_served_users = recommendations_df.groupby("USER").size().mean()
avg_recommendations_for_all_test_users = recommendations_df.groupby("USER").size().reindex(test_user_ids, fill_value=0).mean()

print("Total generated recommendations:", len(recommendations_df))
print("Users receiving at least one recommendation:", users_with_recommendations)
print(f"Average recommendations per served user: {avg_recommendations_for_served_users:.2f}")
print(f"Average recommendations per all test users: {avg_recommendations_for_all_test_users:.2f}")

display(recommendations_df.head(10))

Total generated recommendations: 53411
Users receiving at least one recommendation: 864
Average recommendations per served user: 61.82
Average recommendations per all test users: 53.41


,USER,COURSE_ID,TITLE,SCORE
0,37465,RP0105EN,analyzing big data in r using apache spark,27.0
1,37465,SC0103EN,spark overview for scala analytics,27.0
2,37465,TMP0105EN,getting started with the data apache spark ma...,27.0
3,37465,excourse73,analyzing big data with sql,27.0
4,37465,excourse31,cloud computing applications part 2 big data...,27.0
5,37465,BD0212EN,spark fundamentals ii,27.0
6,37465,excourse72,foundations for big data analysis with sql,27.0
7,37465,excourse70,big data capstone project,24.0
8,37465,excourse03,nosql systems,24.0
9,37465,BD0141EN,accessing hadoop data using hive,24.0


## Most Frequently Recommended Courses

In [5]:
top_recommended = (
    recommendations_df["COURSE_ID"]
    .value_counts()
    .head(10)
    .rename_axis("COURSE_ID")
    .reset_index(name="recommendation_count")
    .merge(course_genres_df[["COURSE_ID", "TITLE"]], on="COURSE_ID", how="left")
)

display(top_recommended)

,COURSE_ID,recommendation_count,TITLE
0,TA0106EN,608,text analytics at scale
1,GPXX0IBEN,548,data science in insurance basic statistical a...
2,excourse22,547,introduction to data science in python
3,excourse21,547,applied machine learning in python
4,ML0122EN,544,accelerating deep learning with gpu
5,GPXX0TY1EN,533,performing database operations in the cloudant...
6,excourse04,533,sql for data science
7,excourse06,533,sql for data science capstone project
8,excourse31,524,cloud computing applications part 2 big data...
9,excourse73,516,analyzing big data with sql


## Interpretation

This method gives broad coverage because user profiles can match many genre-compatible courses. It is useful for discovery and for cold-start situations where there is not enough rating history for collaborative filtering.